In [ ]:
from globals import *
from cpfunctions import *

In [ ]:
# Read file.
df = pd.read_csv(FILE_PATH)

# Encode labels into integers.
le = LabelEncoder()
df["class"] = le.fit_transform(df["class"])

# Get features, class
y = df["class"]

# Drop the label.
X = df.drop(["class"], axis = 1)

# Get column names.
colnames = list(X.columns)

# Save classes order.
save_classes_order(pd.DataFrame(le.classes_), DATASET_PATH)

      class    v1_mfcc1  v1_mfcc2  v1_mfcc3   v1_mfcc4  v1_mfcc5  v1_mfcc6  \
0  watch_tv  120.078886  3.025171  0.390508  -1.777848 -9.216060 -6.567632   
1  watch_tv  130.243139  2.552278 -6.457879  -7.440159 -3.070106 -3.475476   
2  watch_tv  122.259209  1.970267 -8.980575 -10.222030 -3.564994 -2.496906   
3  watch_tv  122.609566  3.188740 -7.049397 -10.569875 -0.220824 -0.649459   
4  watch_tv  121.393023  1.338798 -8.706183  -8.654997 -4.935310 -3.401656   

   v1_mfcc7  v1_mfcc8  v1_mfcc9  ...   v2_maxX   v2_maxY   v2_maxZ  v2_corXY  \
0 -9.918761 -9.145534 -4.625030  ...  0.097412  1.088135  0.227051  0.900163   
1 -5.718819 -2.878174 -0.894522  ...  0.703613  1.424805  0.818115 -0.210431   
2 -5.584900 -6.730588 -4.079890  ...  0.312500  0.972656 -0.070312 -0.018862   
3 -4.975825 -6.856288 -5.524964  ...  0.349854  1.020020 -0.053223  0.492439   
4 -5.031361 -7.957924 -8.613957  ...  0.322266  0.955078 -0.076904 -0.226296   

   v2_corXZ  v2_corYZ  v2_meanMagnitude  v2_sdMagn

In [5]:
# Arrays to store results.
it = []
method = []
groundTruth = []
prediction = []
predictionSet = []
scores = [] # prediction scores as produced by the underlying model.
pvalues = []

for i in range(ITERATIONS):

    curr_it = 1 + i # current iteration.
    
    print("Iteration " + str(curr_it))

    # Set random seed. This should be updated based on iteration number.
    random_seed = 100 + curr_it

    # Split into train, calibration, and test sets.
    X_train, X_rest, y_train, y_rest = train_test_split(X, y,
                                                        train_size = PCT_TRAIN, 
                                                        stratify = y, 
                                                        random_state = random_seed)

    X_calib, X_test, y_calib, y_test = train_test_split(X_rest, y_rest,
                                                        train_size = PCT_CALIBRATION,
                                                        stratify = y_rest,
                                                        random_state = random_seed)


    # Fit the models
    classifiers = fit_models_2v(X_train, y_train, random_seed)
    
    # Build the conformal models.
    for model in classifiers:
        
        modeltype = model[0]
        
        cp = MapieClassifier(estimator=model[1],
                               cv="prefit",
                               method="score",
                               random_state=random_seed,
                               n_jobs=NUMCORES)

        if modeltype in ['v1','v2','v3']:
            selectedcols = [x for x in colnames if modeltype + '_' in x]
        elif modeltype == "aggregated":
            selectedcols = colnames
        elif modeltype == "mvcs":
            selectedcols = colnames
    
        cp.fit(X_calib.loc[:, selectedcols], y_calib)

        y_pred, y_set = cp.predict(X_test.loc[:, selectedcols], alpha=ALPHA)

        y_set = np.squeeze(y_set)

        #### Append results ####
        n = len(y_pred)

        # Iteration
        tmp = np.empty(n, dtype=int)
        tmp.fill(curr_it)
        it.extend(tmp)

        # Method name
        method.extend([model[0]] * n)

        # Ground truth
        groundTruth.extend(le.inverse_transform(y_test))

        # Prediction
        prediction.extend(le.inverse_transform(y_pred))

        # Prediction set.
        predictionSet.extend(["|".join(le.classes_[y_set[i]]) for i in range(n)])

        # Predicted scores.
        pred_scores = model[1].predict_proba(X_test.loc[:, selectedcols])
        scores.extend(["|".join(pred_scores[i,y_set[i]].astype(str)) for i in range(n)])
        
        # Compute p-values.
        cal_probs = model[1].predict_proba(X_calib.loc[:, selectedcols])
        prob_true_class = cal_probs[np.arange(len(X_calib.loc[:, selectedcols])),y_calib]
        calib_scores = 1 - prob_true_class
        test_scores = 1 - pred_scores
        arr_pvalues = compute_pvalues(calib_scores, test_scores)
        pvalues.extend(["|".join(arr_pvalues[i,:].astype(str)) for i in range(n)])

# Store results in data frame.
d = {'it': it, 'method': method, 
     'groundTruth': groundTruth,
     'prediction': prediction,
     'predictionSet': predictionSet,
     'scores': scores,
     'pvalues': pvalues}

results = pd.DataFrame(d)

save_df(results, DATASET_PATH, "1", "results.csv")

print("Done!")

Iteration 1
Iteration 2
Iteration 3
Iteration 4
Iteration 5
Iteration 6
Iteration 7
Iteration 8
Iteration 9
Iteration 10
Iteration 11
Iteration 12
Iteration 13
Iteration 14
Iteration 15
Done!


In [10]:
results.head(10)

,it,method,groundTruth,prediction,predictionSet,scores,pvalues
0,1,v1,type_on_keyboard,watch_tv,type_on_keyboard|watch_tv,0.18|0.42,0.008645533141210375|0.008645533141210375|0.03...
1,1,v1,type_on_keyboard,type_on_keyboard,type_on_keyboard,0.64,0.008645533141210375|0.002881844380403458|0.01...
2,1,v1,brush_teeth,brush_teeth,brush_teeth|eat_chips|wash_hands,0.58|0.14|0.24,0.6311239193083573|0.06628242074927954|0.00288...
3,1,v1,wash_hands,brush_teeth,brush_teeth|eat_chips|sweep|wash_hands,0.34|0.14|0.14|0.3,0.26512968299711814|0.06628242074927954|0.0028...
4,1,v1,wash_hands,wash_hands,wash_hands,0.92,0.005763688760806916|0.002881844380403458|0.00...
5,1,v1,eat_chips,eat_chips,brush_teeth|eat_chips|wash_hands,0.18|0.52|0.2,0.1037463976945245|0.5561959654178674|0.002881...
6,1,v1,brush_teeth,watch_tv,brush_teeth|sweep|type_on_keyboard|watch_tv,0.16|0.14|0.22|0.26,0.07780979827089338|0.008645533141210375|0.008...
7,1,v1,brush_teeth,brush_teeth,brush_teeth,0.96,1.0|0.002881844380403458|0.002881844380403458|...
8,1,v1,brush_teeth,mop_floor,mop_floor|sweep|watch_tv,0.26|0.16|0.18,0.037463976945244955|0.037463976945244955|0.18...
9,1,v1,wash_hands,eat_chips,brush_teeth|eat_chips|wash_hands,0.26|0.42|0.24,0.18155619596541786|0.38904899135446686|0.0057...
